In [ ]:
# Check AMD GPU availability
import torch
print(f'ROCm available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
!pip install PyPDF2 -q

In [ ]:
import requests
import time
import concurrent.futures


class TextAnalyzer:
    def __init__(self, model_name="qwen3-coder:30b", api_url="http://open-webui-ollama.open-webui:11434"):
        self.model_name = model_name
        self.api_url = f"{api_url}/api/chat"
        self.headers = {"Content-Type": "application/json"}
        self.timeout = 600

    def _call_llm(self, prompt, temperature=0.3, max_retries=3):
        """Low-level API call function"""

        # 1. Input validity check: intercept empty input
        if not prompt or not str(prompt).strip():
            return "Error: Empty input or invalid characters detected, intercepted."

        payload = {
            "model": self.model_name,
            "messages": [
                # System Role setting: strengthen output constraints
                {"role": "system",
                 "content": "You are a rigorous data processing assistant. Please output the analysis results directly, and absolutely do not include any opening remarks, prefixes, or explanatory nonsense."},
                {"role": "user", "content": prompt}
            ],
            "stream": False,
            "options": {
                "num_ctx": 32768,
                "temperature": temperature
            }
        }

        # 2. Error handling and retry mechanism (API lifecycle management)
        for attempt in range(max_retries):
            try:
                response = requests.post(self.api_url, json=payload, headers=self.headers, timeout=self.timeout)

                # Handle rate limiting with linear backoff strategy
                if response.status_code == 429:
                    wait_time = 3 * (attempt + 1)
                    print(
                        f"    └── API rate limit triggered (429), waiting {wait_time}s before retrying (Attempt {attempt + 1}/{max_retries})...")
                    time.sleep(wait_time)
                    continue

                response.raise_for_status()

                # Parse response
                res_json = response.json()
                if "message" in res_json:
                    content = res_json["message"]["content"].strip()
                else:
                    content = res_json.get("response", "").strip()

                return content.replace("Please output summary:", "").replace("Summary:", "").strip()

            except requests.exceptions.Timeout:
                print(f"    └── API request timeout, retrying (Attempt {attempt + 1}/{max_retries})...")
                time.sleep(2)
            except Exception as e:
                print(f"    └── API call exception: {e} (Attempt {attempt + 1}/{max_retries})...")
                time.sleep(3)

        return f"Error: API call failed, reached maximum retry limit ({max_retries} attempts)."

    def _chunk_text(self, text, chunk_size=4000):
        return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

    def summarize(self, text):
        """1. Summarization (uses cascading summary generation strategy)"""
        if len(text) > 5000:
            chunks = self._chunk_text(text)
            partial_summaries = [self._call_llm(f"Please briefly summarize the following segment (within 200 words):\n{c}") for c in chunks]
            combined_text = "\n".join(partial_summaries)
            prompt = f"Please combine the summaries of the following multiple segments into a coherent final summary (within 300 words):\n{combined_text}"
        else:
            prompt = f"Please provide a concise summary of the following text (within 300 words):\n{text}"
        return self._call_llm(prompt)

    def analyze_sentiment(self, text):
        """
        2. Sentiment Analysis (Prompt iteration: introducing Few-shot prompting)
        Demonstrates the evolution from Zero-shot to Few-shot, standardizing the output format.
        """
        safe_text = text[:5000]
        prompt = f"""
        Please analyze the sentiment of the following text.
        Output strictly in the following Markdown format.

        [Example 1]
        Input: "The content of this course is very detailed, the teacher is interesting, and it completely exceeded my expectations!"
        Output:
        - **Sentiment**: Positive
        - **Intensity**: 5/5
        - **Brief Reason**: Expresses strong praise and high satisfaction.

        [Example 2]
        Input: "Today's dataset has some missing values and requires time to clean, but it can generally run through."
        Output:
        - **Sentiment**: Neutral
        - **Intensity**: 3/5
        - **Brief Reason**: States an objective problem encountered, but the emotion is stable without obvious negative feelings.

        [Current Task]
        Input: "{safe_text}"
        Output:
        """
        return self._call_llm(prompt)

    def extract_keywords(self, text):
        """
        3. Keyword Extraction (Prompt iteration: introducing Few-shot prompting)
        """
        safe_text = text[:5000]
        prompt = f"""
        Please extract 5-8 core keywords, separated by commas, and output the words directly.

        [Example]
        Input: "Large Language Models (LLMs) have shown amazing emergent abilities in Natural Language Processing tasks, such as in-context learning and chain-of-thought reasoning."
        Output: Large Language Model, Natural Language Processing, Emergent Abilities, In-context Learning, Chain-of-thought

        [Current Task]
        Input: "{safe_text}"
        Output:
        """
        return self._call_llm(prompt)

    def compare_multiple_texts(self, files_dict):
        """
        4. Advanced feature upgrade: Supports comprehensive comparative analysis of N documents (Multi-document comparison)
        - Introduces Dynamic Context Allocation algorithm
        """
        num_files = len(files_dict)
        if num_files < 2:
            return "Error: Comparative analysis requires at least 2 files."

        # [Core Algorithm] Dynamic truncation strategy: The maximum safe context for the LLM is about 24,000 characters.
        # Dynamically distribute the context limit evenly based on the number of files to prevent out-of-memory (OOM) errors.
        max_chars_per_file = 24000 // num_files

        docs_content = ""
        for name, text in files_dict.items():
            # Dynamically extract the first N characters
            safe_text = text[:max_chars_per_file]
            docs_content += f"\n\n====================\n[Document: {name}]\n{safe_text}"

        # Prompt iteration: Dynamic instructions supporting N documents
        prompt = f"""
        Please conduct an in-depth comprehensive comparative analysis of the following {num_files} documents.

        [Chain-of-Thought Guidance]
        Please follow these steps for reasoning:
        1. Summarize the core theme of each document separately.
        2. Find the commonalities of all documents (what common topic are they discussing?).
        3. Deeply compare the main differences in their viewpoints, focuses, or sentiment tendencies.

        [Output Format Restriction]
        Please do not output your thought process. Output the final report directly in the following strict Markdown format:

        ### 📑 1. Core Theme Summary
        (Please list a one-sentence summary for each of these {num_files} documents)

        ### 🤝 2. Commonalities Extraction
        [List core commonalities]

        ### ⚖️ 3. Core Differences Comparison
        - **Dimension 1 ([Extract a difference dimension])**: Document A shows..., Document B shows..., Document C...
        - **Dimension 2 ([Extract a second difference dimension])**: Different emphasis on... among the documents...

        ---
        [The following is the text content of the multiple documents]:
        {docs_content}
        """
        return self._call_llm(prompt)

    def generate_full_report_parallel(self, text):
        """Generate report concurrently"""
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
            future_summary = executor.submit(self.summarize, text)
            future_sentiment = executor.submit(self.analyze_sentiment, text)
            future_keywords = executor.submit(self.extract_keywords, text)

            summary = future_summary.result()
            sentiment = future_sentiment.result()
            keywords = future_keywords.result()

        return summary, sentiment, keywords

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import time
import io
import os
import PyPDF2


# 1. Initialize backend
analyzer = TextAnalyzer(model_name="qwen3-coder:30b")

# 2. Build UI components
header = widgets.HTML("<h2> Intelligent Text Analysis Pipeline </h2>")

mode_selector = widgets.RadioButtons(
    options=['Standard Batch Pipeline (Extract Summary/Sentiment/Keywords)', 'Multi-Document Comprehensive Comparative Analysis (Supports 2 or more)'],
    description='Run Mode:',
    layout=widgets.Layout(width='100%', margin='0 0 15px 0')
)

# Tab 1: Text Input
input_text_area = widgets.Textarea(placeholder='Paste long text here...', layout=widgets.Layout(width='98%', height='150px'))
tab1 = widgets.VBox([widgets.Label("Direct Input (Single Text):"), input_text_area])

# Tab 2: Browser Upload
file_uploader = widgets.FileUpload(accept='.txt, .pdf', multiple=True, description='Browser Upload')
upload_preview_out = widgets.Output(layout={'border': '1px dashed #ccc', 'padding': '10px', 'margin-top': '10px'})
tab2 = widgets.VBox([
    widgets.Label("Note: Due to Jupyter memory limits, large files or multiple PDFs may cause browser upload to fail."),
    file_uploader,
    upload_preview_out
])

# Tab 3: Direct Folder Read
folder_path_input = widgets.Text(
    placeholder='e.g., ./my_pdfs or C:/data/papers',
    description='Folder Path:',
    layout=widgets.Layout(width='80%')
)
btn_scan_folder = widgets.Button(description='Scan This Folder', button_style='info')
folder_preview_out = widgets.Output(layout={'border': '1px dashed #ccc', 'padding': '10px', 'margin-top': '10px'})
tab3 = widgets.VBox([
    widgets.Label("Directly read all TXT/PDF files from a local/server folder."),
    widgets.HBox([folder_path_input, btn_scan_folder]),
    folder_preview_out
])

# Assemble Tabs
tabs = widgets.Tab(children=[tab1, tab2, tab3])
tabs.set_title(0, '📝 Text Paste')
tabs.set_title(1, '📂 Browser Upload (Small Files)')
tabs.set_title(2, '📁 Direct Folder Read (Recommended/Large Batch)')

# Control Area
btn_run = widgets.Button(description='Start Pipeline', button_style='primary',
                         layout=widgets.Layout(width='100%', height='50px', margin='15px 0 0 0'), icon='bolt')
progress = widgets.FloatProgress(value=0.0, min=0.0, max=100.0, description='Ready', bar_style='info',
                                 layout=widgets.Layout(width='100%', visibility='hidden'))
out = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '15px', 'margin-top': '10px'})


# 3. Core parsing logic
def extract_text_from_bytes(name, content_bytes):
    """Generic byte stream extraction function (Supports TXT and PDF)"""
    if name.lower().endswith('.pdf'):
        try:
            pdf_reader = PyPDF2.PdfReader(io.BytesIO(content_bytes))
            if pdf_reader.is_encrypted: return "⚠️ [Parsing Failed] PDF is encrypted"
            extracted_text = ""
            for page in pdf_reader.pages:
                try:
                    text = page.extract_text()
                    if text: extracted_text += text + "\n"
                except:
                    pass
            if not extracted_text.strip(): return "⚠️ [Parsing Warning] Extraction empty, might be a scanned image only"
            return extracted_text
        except Exception as e:
            return f"⚠️ [Parsing Crash] PDF corrupted ({e})"
    else:
        try:
            return content_bytes.decode("utf-8")
        except Exception as e:
            return f"⚠️ [Encoding Error] TXT must be UTF-8 ({e})"


def render_dashboard(files_dict, output_widget):
    """Render status dashboard"""
    with output_widget:
        clear_output()
        if not files_dict:
            display(Markdown("No valid files detected"))
            return

        table_md = "| File Name | Status | Extracted Characters | Estimated Tokens (Approx.) |\n| :--- | :---: | :---: | :---: |\n"
        total_chars, total_tokens, valid_files = 0, 0, 0

        for name, text in files_dict.items():
            if text.startswith("⚠️"):
                table_md += f"| `{name}` | Error | 0 | 0 |\n  > *{text}*\n"
            else:
                chars = len(text)
                tokens = int(chars * 0.7)
                total_chars += chars
                total_tokens += tokens
                valid_files += 1
                table_md += f"| `{name}` | Success | {chars:,} chars | ~{tokens:,} |\n"

        summary = f"### 🎛️ Real-time File Dashboard\n- **Successfully Extracted**: {valid_files} files\n- **Total Text Volume**: {total_chars:,} characters (Estimated to consume {total_tokens:,} Tokens)\n\n"
        if total_tokens > 24000:
            summary += "> **Smart Alert**: High total Token count detected, automatically triggering long-text dimensionality reduction strategy.\n\n"
        display(Markdown(summary + table_md))


# 4. Listener bindings
# Listen to Tab 2: Browser Upload
def on_file_upload_change(change):
    if not file_uploader.value: return
    with upload_preview_out:
        print("⏳ Scanning and parsing documents, please wait...")
    files_dict = {}
    items = file_uploader.value if isinstance(file_uploader.value, tuple) else file_uploader.value.values()
    for info in items:
        name = info['name'] if isinstance(info, dict) else info.name
        raw_data = info['content'] if isinstance(info, dict) else info.content
        b_data = raw_data.tobytes() if isinstance(raw_data, memoryview) else bytes(raw_data)
        files_dict[name] = extract_text_from_bytes(name, b_data)
    render_dashboard(files_dict, upload_preview_out)


file_uploader.observe(on_file_upload_change, names='value')

# Listen to Tab 3: Scan local folder
global_folder_files = {}  # Store content read from folder


def on_scan_folder_click(b):
    folder_path = folder_path_input.value.strip()
    global global_folder_files
    global_folder_files.clear()

    with folder_preview_out:
        clear_output()
        if not folder_path or not os.path.exists(folder_path):
            print(f"Folder not found: {folder_path} (Please ensure the path is correct; use ./xxx for current directory)")
            return
        print(f"⏳ Reading local folder: {folder_path} ...")

    for filename in os.listdir(folder_path):
        if filename.lower().endswith('.pdf') or filename.lower().endswith('.txt'):
            file_path = os.path.join(folder_path, filename)
            try:
                with open(file_path, 'rb') as f:
                    content_bytes = f.read()
                global_folder_files[filename] = extract_text_from_bytes(filename, content_bytes)
            except Exception as e:
                global_folder_files[filename] = f"⚠️ Unable to read file ({e})"

    render_dashboard(global_folder_files, folder_preview_out)


btn_scan_folder.on_click(on_scan_folder_click)


# 5. Core startup logic
def on_click_analysis(b):
    out.clear_output()
    btn_run.disabled = True
    progress.layout.visibility = 'visible'
    progress.value = 5
    progress.bar_style = 'info'

    current_tab = tabs.selected_index
    start_time = time.time()
    texts_to_process = {}

    try:
        # Get text sources to process
        if current_tab == 0:
            if not input_text_area.value.strip(): raise Exception("Text content is empty!")
            texts_to_process["Manually entered text"] = input_text_area.value
        elif current_tab == 1:
            if not file_uploader.value: raise Exception("Please upload files first! (Or try switching to Tab 3 to read a folder)")
            items = file_uploader.value if isinstance(file_uploader.value, tuple) else file_uploader.value.values()
            for info in items:
                name = info['name'] if isinstance(info, dict) else info.name
                raw_data = info['content'] if isinstance(info, dict) else info.content
                b_data = raw_data.tobytes() if isinstance(raw_data, memoryview) else bytes(raw_data)
                texts_to_process[name] = extract_text_from_bytes(name, b_data)
        elif current_tab == 2:
            if not global_folder_files: raise Exception("Please enter the path and click [Scan This Folder] first!")
            texts_to_process = global_folder_files.copy()

        # Filter out invalid files with ⚠️ warnings
        valid_files = {k: v for k, v in texts_to_process.items() if not v.startswith("⚠️")}

        if 'Comparative Analysis' in mode_selector.value:
            if len(valid_files) < 2: raise Exception(f"Comparative analysis requires at least 2 valid files, currently {len(valid_files)} valid!")
            progress.value = 20
            progress.description = 'Comparing...'
            with out:
                print(f"⏳ Comparing {len(valid_files)} documents, please wait...")
            compare_result = analyzer.compare_multiple_texts(valid_files)

            progress.value = 100
            with out:
                clear_output()
                display(Markdown(
                    f"# Multi-Document Comprehensive Comparison Report\n> ⏱️ **Total Time**: {time.time() - start_time:.2f}s\n\n{compare_result}"))

        else:
            if not valid_files: raise Exception("No valid files detected, please check the dashboard for errors.")
            total_files = len(valid_files)
            all_reports = []

            for idx, (filename, text_content) in enumerate(valid_files.items()):
                progress.description = f'Processing ({idx + 1}/{total_files})'
                with out: print(f"⏳ Processing [{idx + 1}/{total_files}]: {filename} ...")

                summary, sentiment, keywords = analyzer.generate_full_report_parallel(text_content)
                all_reports.append(
                    f"## 📄 {filename}\n- **🔑 Summary**: {summary}\n- **🎭 Sentiment**: \n{sentiment}\n- **🏷️ Keywords**: {keywords}\n---")
                progress.value = 10 + (90 / total_files) * (idx + 1)

            progress.value = 100
            with out:
                clear_output()
                display(Markdown(
                    f"# 📊 Batch Analysis Complete\n> 📑 **Successfully Processed**: {total_files} files\n> ⏱️ **Total Time**: {time.time() - start_time:.2f} seconds\n\n" + "\n".join(
                        all_reports)))

        progress.description = 'Complete!'
        progress.bar_style = 'success'

    except Exception as e:
        with out:
            display(Markdown(f"**Error Occurred**: {e}"))
        progress.bar_style = 'danger'
    finally:
        btn_run.disabled = False


btn_run.on_click(on_click_analysis)

# --- 6. Display UI ---
ui = widgets.VBox([header, mode_selector, tabs, btn_run, progress, out])
display(ui)